In [ ]:
from pathlib import Path
import numpy as np
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('fivethirtyeight')
from matplotlib.ticker import FuncFormatter
from nltk.corpus import stopwords
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import CountVectorizer
import os

In [ ]:
### cell 0 ###

factor = 50000
train = pd.read_csv(f"{Path(__file__).parent.parent}/input/feedback-prize-2021/train.csv")
train = train[:factor]    

train[['discourse_id', 'discourse_start', 'discourse_end']] = train[['discourse_id', 'discourse_start', 'discourse_end']].astype(int)

sample_submission = pd.read_csv(f"{Path(__file__).parent.parent}/input/feedback-prize-2021/sample_submission.csv", dtype={"class": "float64", "predictionstring": "float64"})

#The glob module finds all the pathnames matching a specified pattern according to the rules used by the Unix shell
train_txt = glob(f"{Path(__file__).parent.parent}/input/feedback-prize-2021/train/*.txt")
test_txt = glob(f"{Path(__file__).parent.parent}/input/feedback-prize-2021/test/*.txt")
sample_submission.info()

In [ ]:
### cell 1 ###

sample_discourse_id = train.query('id == "423A1CA112E2"').iloc[0]["discourse_id"]

In [ ]:
### cell 2 ###

#add columns
train["discourse_len"] = train["discourse_text"].apply(lambda x: len(x.split()))
train["pred_len"] = train["predictionstring"].apply(lambda x: len(x.split()))

cols_to_display = ['discourse_id', 'discourse_text', 'discourse_type','predictionstring', 'discourse_len', 'pred_len']
train[cols_to_display].head()

In [ ]:
### cell 3 ###

print(f"The total number of discourses is {len(train)}")
train.query('discourse_len != pred_len')[cols_to_display]

In [ ]:
### cell 4 ###

print(train.query(f'discourse_id == {sample_discourse_id}')['discourse_text'].values[0])
print(train.query(f'discourse_id == {sample_discourse_id}')['discourse_text'].values[0].split())
print(len(train.query(f'discourse_id == {sample_discourse_id}')['discourse_text'].values[0].split()))

In [ ]:
### cell 5 ###

print(train.query(f'discourse_id == {sample_discourse_id}')['predictionstring'].values[0])
print(train.query(f'discourse_id == {sample_discourse_id}')['predictionstring'].values[0].split())
print(len(train.query(f'discourse_id == {sample_discourse_id}')['predictionstring'].values[0].split()))

In [ ]:
### cell 6 ###

ax1 = train.groupby('discourse_type')['discourse_len'].mean().sort_values()
ax2 = train.groupby('discourse_type')['discourse_type'].count().sort_values()

In [ ]:
### cell 7 ###

av_per_essay = train['discourse_type_num'].value_counts(ascending = True)
av_per_essay.index.name = "discourse_type_num"
av_per_essay = av_per_essay.reset_index(name='count')

av_per_essay['perc'] = round((av_per_essay['count'] / train.id.nunique()),3)
av_per_essay = av_per_essay.set_index('discourse_type_num')
ax = av_per_essay.query('perc > 0.03')['perc']

In [ ]:
### cell 8 ###

data = train.groupby("discourse_type")[['discourse_end', 'discourse_start']].mean().reset_index().sort_values(by = 'discourse_start', ascending = False)

In [ ]:
### cell 9 ###

train_first = train.drop_duplicates(subset = "id", keep = "first").discourse_type.value_counts()
train_first.index.name = 'discourse_type'
train_first = train_first.reset_index(name='counts_first')

train_first['percent_first'] = round((train_first['counts_first']/train.id.nunique()),2)
train_last = train.drop_duplicates(subset = "id", keep = "last").discourse_type.value_counts()
train_last.index.name = 'discourse_type'
train_last = train_last.reset_index(name='counts_last')


train_last['percent_last'] = round((train_last['counts_last']/train.id.nunique()),2)
train_first_last = train_first.merge(train_last, on = "discourse_type", how = "left")
train_first_last

In [ ]:
### cell 10 ###

train['discourse_nr'] = 1
counter = 1

for i_1 in tqdm(range(1, len(train))):
    if train.loc[i_1, 'id'] == train.loc[i_1-1, 'id']:
        counter += 1
        train.loc[i_1, 'discourse_nr'] = counter
    else:
        counter = 1
        train.loc[i_1, 'discourse_nr'] = counter

train.query('discourse_type in ["Lead"]').groupby('discourse_type_num')['discourse_nr']

In [ ]:
### cell 11 ###

# this code chunk is copied from Rob Mulla
len_dict = {}
word_dict = {}
for t in tqdm(train_txt):
    with open(t, "r") as txt_file:
        myid = t.split("/")[-1].replace(".txt", "")
        data = txt_file.read()
        mylen = len(data.strip())
        myword = len(data.split())
        len_dict[myid] = mylen
        word_dict[myid] = myword
train["essay_len"] = train["id"].map(len_dict)
train["essay_words"] = train["id"].map(word_dict)

In [ ]:
### cell 12 ###

#initialize column
train['gap_length'] = np.nan

#set the first one
train.loc[0, 'gap_length'] = 7 #discourse start - 1 (previous end is always -1)

#loop over rest
for i_3 in tqdm(range(1, len(train))):
    #gap if difference is not 1 within an essay
    if ((train.loc[i_3, "id"] == train.loc[i_3-1, "id"])\
        and (train.loc[i_3, "discourse_start"] - train.loc[i_3-1, "discourse_end"] > 1)):
        train.loc[i_3, 'gap_length'] = train.loc[i_3, "discourse_start"] - train.loc[i_3-1, "discourse_end"] - 2
        #minus 2 as the previous end is always -1 and the previous start always +1
    #gap if the first discourse of an new essay does not start at 0
    elif ((train.loc[i_3, "id"] != train.loc[i_3-1, "id"])\
        and (train.loc[i_3, "discourse_start"] != 0)):
        train.loc[i_3, 'gap_length'] = train.loc[i_3, "discourse_start"] -1


 #is there any text after the last discourse of an essay?
last_ones = train.drop_duplicates(subset="id", keep='last')
last_ones['gap_end_length'] = np.where((last_ones.discourse_end < last_ones.essay_len),\
                                       (last_ones.essay_len - last_ones.discourse_end),\
                                       np.nan)

cols_to_merge = ['id', 'discourse_id', 'gap_end_length']
train = train.merge(last_ones[cols_to_merge], on = ["id", "discourse_id"], how = "left")

In [ ]:
### cell 13 ###

#display an example
cols_to_display = ['id', 'discourse_start', 'discourse_end', 'discourse_type', 'essay_len', 'gap_length', 'gap_end_length']
train[cols_to_display].query('id == "AFEC37C2D43F"')

In [ ]:
### cell 14 ###

#how many pieces of tekst are not used as discourses?
print(f"Besides the {len(train)} discourse texts, there are {len(train.query('gap_length.notna()', engine='python'))+ len(train.query('gap_end_length.notna()', engine='python'))} pieces of text not classified.")

In [ ]:
### cell 15 ###

IREWR_tmp = train.sort_values(by = "gap_length", ascending = False)[cols_to_display].head()
IREWR_plug_2 = IREWR_tmp.iloc[0]["id"]
IREWR_tmp

In [ ]:
### cell 16 ###

IREWR_tmp2 = train.sort_values(by = "gap_end_length", ascending = False)[cols_to_display].head()
IREWR_plug_1 = IREWR_tmp2.iloc[1]["id"]
IREWR_tmp2

In [ ]:
### cell 17 ###

all_gaps = (train.gap_length[~train.gap_length.isna()])._append((train.gap_end_length[~train.gap_end_length.isna()]), ignore_index= True)
all_gaps = all_gaps[all_gaps<300]

In [ ]:
### cell 18 ###

total_gaps = train.groupby('id').agg({'essay_len': 'first',\
                                               'gap_length': 'sum',\
                                               'gap_end_length': 'sum'})
total_gaps['perc_not_classified'] = round(((total_gaps.gap_length + total_gaps.gap_end_length)/total_gaps.essay_len),2)

total_gaps.sort_values(by = 'perc_not_classified', ascending = False).head()

In [ ]:
### cell 19 ###

def add_gap_rows(essay):
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop = True)
    
    print(df_essay)

    #index new row
    insert_row = len(df_essay)
   
    for i_2 in range(1, len(df_essay)):          
        if df_essay.loc[i_2,"gap_length"] >0:
            if i_2 == 0:
                start = 0 #as there is no i-1 for first row
                end = df_essay.loc[0, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1
            else:
                start = df_essay.loc[i_2-1, "discourse_end"] + 1
                end = df_essay.loc[i_2, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1

    df_essay = df_essay.sort_values(by = "discourse_start").reset_index(drop=True)

    #add gap at end
    if df_essay.loc[(len(df_essay)-1),'gap_end_length'] > 0:
        start = df_essay.loc[(len(df_essay)-1), "discourse_end"] + 1
        end = start + df_essay.loc[(len(df_essay)-1), 'gap_end_length']
        disc_type = "Nothing"
        gap_end = np.nan
        gap = np.nan
        df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
        
    return(df_essay)
add_gap_rows(IREWR_plug_1)

In [ ]:
### cell 20 ###

def add_gap_rows_2(essay):
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop = True)
    
    print(df_essay)

    #index new row
    insert_row = len(df_essay)
   
    for i_2 in range(1, len(df_essay)):          
        if df_essay.loc[i_2,"gap_length"] >0:
            if i_2 == 0:
                start = 0 #as there is no i-1 for first row
                end = df_essay.loc[0, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1
            else:
                start = df_essay.loc[i_2-1, "discourse_end"] + 1
                end = df_essay.loc[i_2, 'discourse_start'] -1
                disc_type = "Nothing"
                gap_end = np.nan
                gap = np.nan
                df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
                insert_row += 1

    df_essay = df_essay.sort_values(by = "discourse_start").reset_index(drop=True)

    #add gap at end
    if df_essay.loc[(len(df_essay)-1),'gap_end_length'] > 0:
        start = df_essay.loc[(len(df_essay)-1), "discourse_end"] + 1
        end = start + df_essay.loc[(len(df_essay)-1), 'gap_end_length']
        disc_type = "Nothing"
        gap_end = np.nan
        gap = np.nan
        df_essay.loc[insert_row] = [start, end, disc_type, gap, gap_end]
        
    return(df_essay)
    
def print_colored_essay(essay):
    df_essay = add_gap_rows_2(essay)
    #code from https://www.kaggle.com/odins0n/feedback-prize-eda, but adjusted to df_essay
    essay_file = f"{Path(__file__).parent.parent}/input/feedback-prize-2021/train/" + essay + ".txt"

    ents = []
    for i, row in df_essay.iterrows():
        ents.append({
                        'start': int(row['discourse_start']), 
                         'end': int(row['discourse_end']), 
                         'label': row['discourse_type']
                    })

    with open(essay_file, 'r') as file: data = file.read()

    doc2 = {
        "text": data,
        "ents": ents,
    }

    colors = {'Lead': '#EE11D0','Position': '#AB4DE1','Claim': '#1EDE71','Evidence': '#33FAFA','Counterclaim': '#4253C1','Concluding Statement': 'yellow','Rebuttal': 'red'}
    options = {"ents": df_essay.discourse_type.unique().tolist(), "colors": colors}

print_colored_essay(IREWR_plug_2)

In [ ]:
### cell 21 ###

train['discourse_text'] = train['discourse_text'].str.lower()

#get stopwords from nltk library
stop_english = stopwords.words("english")
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

#put dataframe of Top-10 words in dict for all discourse types
counts_dict = {}
for dt in train['discourse_type'].unique():
    df = train.query(f"discourse_type == '{dt}'")
    text = df.discourse_text.apply(lambda x: x.split()).tolist()
    text = [item for elem in text for item in elem]
    df1 = pd.Series(text).value_counts().to_frame().reset_index()
    df1.columns = ['Word', 'Frequency']
    df1 = df1[~df1.Word.isin(stop_english)].head(10)
    df1 = df1.set_index("Word").sort_values(by = "Frequency", ascending = True)
    counts_dict[dt] = df1

keys = list(counts_dict.keys())

In [ ]:
### cell 22 ###

def get_n_grams(n_grams, top_n = 10):
    df_words = pd.DataFrame()
    for dt in tqdm(train['discourse_type'].unique()):
        df = train.query(f"discourse_type == '{dt}'")
        texts = df['discourse_text'].tolist()
        vec = CountVectorizer(lowercase = True, stop_words = 'english',\
                              ngram_range=(n_grams, n_grams)).fit(texts)
        bag_of_words = vec.transform(texts)
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
        cvec_df = pd.DataFrame.from_records(words_freq,\
                                            columns= ['words', 'counts']).sort_values(by="counts", ascending=False)
        cvec_df.insert(0, "Discourse_type", dt)
        cvec_df = cvec_df.iloc[:top_n,:]
        df_words = df_words._append(cvec_df)
    return df_words
bigrams = get_n_grams(n_grams = 2, top_n=10)
bigrams.head()

In [ ]:
### cell 23 ###

def plot_ngram(df, type = "bigrams"):
    for n, dt in enumerate(df.Discourse_type.unique()):
        data = df.query('Discourse_type == @dt')[['words', 'counts']].set_index("words").sort_values(by = "counts", ascending = True)
    
plot_ngram(bigrams)

In [ ]:
### cell 24 ###

def get_n_grams_2(n_grams, top_n = 10):
    df_words = pd.DataFrame()
    for dt in tqdm(train['discourse_type'].unique()):
        df = train.query(f"discourse_type == '{dt}'")
        texts = df['discourse_text'].tolist()
        vec = CountVectorizer(lowercase = True, stop_words = 'english',\
                              ngram_range=(n_grams, n_grams)).fit(texts)
        bag_of_words = vec.transform(texts)
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
        cvec_df = pd.DataFrame.from_records(words_freq,\
                                            columns= ['words', 'counts']).sort_values(by="counts", ascending=False)
        cvec_df.insert(0, "Discourse_type", dt)
        cvec_df = cvec_df.iloc[:top_n,:]
        df_words = df_words._append(cvec_df)
    return df_words
    
def plot_ngram_2(df, type = "bigrams"):
    data = df.query('Discourse_type == @dt')[['words', 'counts']].set_index("words").sort_values(by = "counts", ascending = True)
    
trigrams = get_n_grams_2(n_grams = 3, top_n=10)
plot_ngram_2(trigrams, type = "trigrams")

In [ ]:
### cell 25 ###

# https://www.kaggle.com/raghavendrakotala/fine-tunned-on-roberta-base-as-ner-problem-0-533
test_names, train_texts = [], []
for f in tqdm(list(os.listdir(f"{Path(__file__).parent.parent}/input/feedback-prize-2021/train"))):
    test_names.append(f.replace('.txt', ''))
    train_texts.append(open(f"{Path(__file__).parent.parent}/input/feedback-prize-2021/train/" + f, 'r').read())
train_text_df = pd.DataFrame({'id': test_names, 'text': train_texts})
train_text_df.head()

In [ ]:
### cell 26 ###

all_entities = []

#loop over dataframe with all full texts
for i_4 in tqdm(range(len(train_text_df))):
    total = len(train_text_df.loc[i_4, 'text'].split())
    #now a list with length the total number of words in an essay is initialised with all values being "O"
    entities = ["O"]*total
    #now loop over dataframe with all discourses of this particular essay
    discourse_id = train_text_df.loc[i_4, 'id']
    train_df_id = train.query(f"id == '{discourse_id}'").reset_index(drop=True)
    for j_4 in range(len(train_df_id)):
        discourse = train_df_id.loc[j_4, 'discourse_type']
        #make a list with the position numbers in predictionstring converted into integer
        list_ix = [int(x) for x in train_df_id.loc[j_4, 'predictionstring'].split(' ')]
        #now the entities lists gets overwritten where there are discourse identified by the experts
        #the first word of each discourse gets prefix "Beginning"
        entities[list_ix[0]] = f"B-{discourse}"
        #the other ones get prefix I
        for k in list_ix[1:]: entities[k] = f"I-{discourse}"
    all_entities.append(entities)
    
    
train_text_df['entities'] = all_entities

In [ ]:
### cell 27 ###

train_text_df.head()